# Final Backtest Summary Viewer

Load existing final-evaluation summary CSVs and compare one selected metric across tickers and strategies.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.parent.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# ---- User configuration ----
REPORT_SPECS = [
    ("dynamic", PROJECT_DIR / "reports" / "final_backtest_toxic_cif")
]
PRIMARY_RUN_MODE = "dynamic"  # Existing single-mode tables/plots use this mode.
SELECTED_METRIC = "toxic_cost_mean_bps"
# Other useful options:
# SELECTED_METRIC = "mean_is_bps"
# SELECTED_METRIC = "time_weighted_mean_is_bps"
# SELECTED_METRIC = "toxic_cost_mean_bps"


def load_summary_tree(report_root: Path, run_mode: str) -> tuple[pd.DataFrame, list[Path]]:
    summary_paths = sorted(report_root.glob("*/summary_all_strategies.csv"))
    if not summary_paths:
        summary_paths = sorted(report_root.glob("*/*/summary.csv"))

    frames = []
    for path in summary_paths:
        frame = pd.read_csv(path)
        if frame.empty:
            continue
        if "ticker" not in frame.columns:
            frame["ticker"] = path.parent.name.upper() if path.name == "summary_all_strategies.csv" else path.parent.parent.name.upper()
        if "strategy" not in frame.columns:
            frame["strategy"] = path.parent.name
        frame["run_mode"] = run_mode
        frame["report_root"] = str(report_root)
        frame["source_path"] = str(path)
        frames.append(frame)
    return (pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()), summary_paths

frames = []
path_counts = {}
for run_mode, report_root in REPORT_SPECS:
    frame, paths = load_summary_tree(Path(report_root), run_mode)
    path_counts[run_mode] = len(paths)
    if not frame.empty:
        frames.append(frame)

summary_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if summary_df.empty:
    raise FileNotFoundError(f"No summary CSVs found under configured report roots: {REPORT_SPECS}")

summary_df["ticker"] = summary_df["ticker"].astype(str).str.upper()
summary_df["model_type"] = summary_df.get("strategy", summary_df.get("model_type", "")).astype(str)
summary_df["run_mode"] = summary_df["run_mode"].astype(str)

if PRIMARY_RUN_MODE not in set(summary_df["run_mode"]):
    raise ValueError(f"PRIMARY_RUN_MODE={PRIMARY_RUN_MODE!r} is not loaded. Available: {sorted(summary_df['run_mode'].unique())}")
analysis_df = summary_df[summary_df["run_mode"].eq(PRIMARY_RUN_MODE)].copy()

print(f"Loaded {len(summary_df):,} summary row(s) across {len(REPORT_SPECS)} run mode(s).")
print("CSV files by run mode:", path_counts)
print(f"Primary run mode for single-mode plots: {PRIMARY_RUN_MODE}")
display(summary_df.head())

# Baseline sanity check across run modes.
if SELECTED_METRIC in summary_df.columns:
    baseline_check = summary_df[summary_df["model_type"].eq("baseline")].pivot_table(
        index="ticker",
        columns="run_mode",
        values=SELECTED_METRIC,
        aggfunc="first",
    )
    if baseline_check.shape[1] >= 2:
        baseline_check = baseline_check.apply(pd.to_numeric, errors="coerce")
        baseline_check["max_abs_diff"] = baseline_check.sub(baseline_check.iloc[:, 0], axis=0).abs().max(axis=1)
        print(f"Baseline max abs diff across run modes for {SELECTED_METRIC}: {baseline_check['max_abs_diff'].max():.6g}")
        display(baseline_check)


In [ ]:
if SELECTED_METRIC not in summary_df.columns:
    raise ValueError(
        f"Metric {SELECTED_METRIC!r} is not available. Existing columns include: "
        f"{sorted(summary_df.columns)}"
    )

table = summary_df[["run_mode", "ticker", "model_type", SELECTED_METRIC]].copy()
table[SELECTED_METRIC] = pd.to_numeric(table[SELECTED_METRIC], errors="coerce")
table = table.sort_values(["run_mode", "ticker", "model_type"]).reset_index(drop=True)

def color_metric_within_ticker(frame: pd.DataFrame) -> pd.DataFrame:
    styles = pd.DataFrame("", index=frame.index, columns=frame.columns)
    for _, group in frame.groupby(["run_mode", "ticker"], sort=False):
        metric_values = pd.to_numeric(group[SELECTED_METRIC], errors="coerce")
        finite = metric_values[np.isfinite(metric_values)]
        if finite.empty:
            continue
        lo = float(finite.min())
        hi = float(finite.max())
        if np.isclose(lo, hi):
            styles.loc[group.index, SELECTED_METRIC] = "background-color: #f2f2f2"
            continue

        for idx, value in metric_values.items():
            if not np.isfinite(value):
                styles.loc[idx, SELECTED_METRIC] = "background-color: #f7f7f7; color: #999999"
                continue
            # Lower IS is better: best row within each ticker/mode is green, worst is red.
            score = (float(value) - lo) / (hi - lo)
            green = int(round(235 - 65 * score))
            red = int(round(255 - 45 * (1.0 - score)))
            styles.loc[idx, SELECTED_METRIC] = f"background-color: rgb({red}, {green}, 210)"
    return styles

styled = (
    table.style
    .format({SELECTED_METRIC: "{:.4f}"})
    .apply(color_metric_within_ticker, axis=None)
    .set_caption(
        f"Selected metric: {SELECTED_METRIC}; lower is better, "
        "colored within each ticker and run mode"
    )
)
display(styled)


In [ ]:
# Grouped bar chart by ticker and strategy.
# If SELECTED_METRIC has bootstrap CI columns, error bars are drawn automatically.
import matplotlib.pyplot as plt

STRATEGY_ORDER = ["baseline", "gru", "gru_transformer", "transformer", "mamba"]
STRATEGY_LABELS = {
    "baseline": "Baseline",
    "gru": "GRU",
    "gru_transformer": "GRU + Trans.",
    "transformer": "Transformer",
    "mamba": "Mamba",
}

METRIC_LABELS = {
    "mean_is_bps": "mean IS, bps",
    "median_is_bps": "median IS, bps",
    "time_weighted_mean_is_bps": "time-weighted mean IS, bps",
    "toxic_cost_mean_bps": "mean toxic cost, bps",
    "bootstrap_mean_is_bps_mean": "bootstrap mean IS, bps",
    "bootstrap_median_is_bps_mean": "bootstrap median IS, bps",
    "bootstrap_time_weighted_mean_is_bps_mean": "bootstrap time-weighted mean IS, bps",
    "bootstrap_toxic_cost_mean_bps_mean": "mean toxic cost, bps",
}


def metric_label(metric: str) -> str:
    return METRIC_LABELS.get(metric, metric.replace("_", " "))

plot_df = analysis_df.copy()
if SELECTED_METRIC not in plot_df.columns:
    raise ValueError(f"Metric {SELECTED_METRIC!r} is not available.")

plot_df[SELECTED_METRIC] = pd.to_numeric(plot_df[SELECTED_METRIC], errors="coerce")
plot_df["model_type"] = plot_df["model_type"].astype(str)
plot_df = plot_df[np.isfinite(plot_df[SELECTED_METRIC])].copy()

tickers = sorted(plot_df["ticker"].dropna().unique())
available_strategies = list(plot_df["model_type"].dropna().unique())
strategies = [s for s in STRATEGY_ORDER if s in available_strategies]
strategies.extend(sorted(s for s in available_strategies if s not in strategies))

ci_low_col = f"bootstrap_{SELECTED_METRIC}_ci_low"
ci_high_col = f"bootstrap_{SELECTED_METRIC}_ci_high"
has_bootstrap_ci = {ci_low_col, ci_high_col}.issubset(plot_df.columns)
if has_bootstrap_ci:
    plot_df[ci_low_col] = pd.to_numeric(plot_df[ci_low_col], errors="coerce")
    plot_df[ci_high_col] = pd.to_numeric(plot_df[ci_high_col], errors="coerce")

x = np.arange(len(tickers))
width = min(0.82 / max(len(strategies), 1), 0.18)
fig_width = max(12, 0.8 * len(tickers) + 4)
fig, ax = plt.subplots(figsize=(fig_width, 6))

palette = plt.get_cmap("tab10")
for j, strategy in enumerate(strategies):
    strat = plot_df[plot_df["model_type"].eq(strategy)].set_index("ticker")
    values = strat.reindex(tickers)[SELECTED_METRIC]
    offset = (j - (len(strategies) - 1) / 2) * width
    yerr = None
    if has_bootstrap_ci:
        lows = strat.reindex(tickers)[ci_low_col]
        highs = strat.reindex(tickers)[ci_high_col]
        lower = (values - lows).clip(lower=0)
        upper = (highs - values).clip(lower=0)
        yerr = np.vstack([lower.to_numpy(dtype=float), upper.to_numpy(dtype=float)])
        yerr[:, ~np.isfinite(yerr).all(axis=0)] = 0.0

    ax.bar(
        x + offset,
        values.to_numpy(dtype=float),
        width=width,
        label=STRATEGY_LABELS.get(strategy, strategy),
        color=palette(j % 10),
        edgecolor="white",
        linewidth=0.6,
        yerr=yerr,
        capsize=2.5 if has_bootstrap_ci else 0,
        error_kw={"elinewidth": 0.9, "alpha": 0.75},
    )

ax.axhline(0.0, color="0.25", linewidth=0.9, linestyle="--", alpha=0.75)
ax.set_xticks(x)
ax.set_xticklabels(tickers, rotation=35, ha="right")
ax.set_xlabel("Ticker")
ax.set_ylabel(metric_label(SELECTED_METRIC))
ci_note = " with bootstrap 95% CI" if has_bootstrap_ci else ""
ax.set_title(f"{PRIMARY_RUN_MODE}: final backtest comparison, {metric_label(SELECTED_METRIC)}{ci_note}")
ax.legend(ncol=min(len(strategies), 5), frameon=False, loc="best")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()


In [ ]:
# Average selected metric across tickers: one bar per strategy.
# Error bars show an approximate 95% CI of the cross-ticker mean.
avg_df = analysis_df.copy()
if SELECTED_METRIC not in avg_df.columns:
    raise ValueError(f"Metric {SELECTED_METRIC!r} is not available.")

avg_df[SELECTED_METRIC] = pd.to_numeric(avg_df[SELECTED_METRIC], errors="coerce")
avg_df["model_type"] = avg_df["model_type"].astype(str)
avg_df = avg_df[np.isfinite(avg_df[SELECTED_METRIC])].copy()

strategy_order = [s for s in STRATEGY_ORDER if s in set(avg_df["model_type"])]
strategy_order.extend(sorted(s for s in avg_df["model_type"].unique() if s not in strategy_order))

avg_summary = (
    avg_df.groupby("model_type")[SELECTED_METRIC]
    .agg(mean="mean", std="std", count="count")
    .reindex(strategy_order)
    .reset_index()
)
avg_summary["sem"] = avg_summary["std"] / np.sqrt(avg_summary["count"])
avg_summary["ci95"] = 1.96 * avg_summary["sem"]

fig, ax = plt.subplots(figsize=(8.5, 5.2))
colors = [plt.get_cmap("tab10")(i % 10) for i in range(len(avg_summary))]
x = np.arange(len(avg_summary))
ax.bar(
    x,
    avg_summary["mean"],
    yerr=avg_summary["ci95"].fillna(0.0),
    capsize=4,
    color=colors,
    edgecolor="white",
    linewidth=0.8,
    error_kw={"elinewidth": 1.0, "alpha": 0.8},
)
ax.axhline(0.0, color="0.25", linewidth=0.9, linestyle="--", alpha=0.75)
ax.set_xticks(x)
ax.set_xticklabels([STRATEGY_LABELS.get(s, s) for s in avg_summary["model_type"]], rotation=20, ha="right")
ax.set_ylabel(metric_label(SELECTED_METRIC))
ax.set_title(f"{PRIMARY_RUN_MODE}: average final backtest performance, {metric_label(SELECTED_METRIC)}")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()

display(avg_summary[["model_type", "mean", "ci95", "count"]])


In [ ]:
# Recommended stability plot: per-ticker improvement vs baseline by model type.
# Positive values mean the model improves over the always-place baseline.
import matplotlib.pyplot as plt

stability_df = analysis_df.copy()
if SELECTED_METRIC not in stability_df.columns:
    raise ValueError(f"Metric {SELECTED_METRIC!r} is not available.")

stability_df[SELECTED_METRIC] = pd.to_numeric(stability_df[SELECTED_METRIC], errors="coerce")
stability_df["model_type"] = stability_df["model_type"].astype(str)
stability_df = stability_df[np.isfinite(stability_df[SELECTED_METRIC])].copy()

baseline = (
    stability_df[stability_df["model_type"].eq("baseline")]
    .set_index("ticker")[SELECTED_METRIC]
    .rename("baseline_metric")
)
models = stability_df[~stability_df["model_type"].eq("baseline")].copy()
models = models.join(baseline, on="ticker")
models = models[np.isfinite(models["baseline_metric"])].copy()
models["improvement_bps"] = models["baseline_metric"] - models[SELECTED_METRIC]
finite_improvements = models["improvement_bps"].replace([np.inf, -np.inf], np.nan).dropna().abs()
linthresh = max(0.01, float(finite_improvements.quantile(0.10))) if not finite_improvements.empty else 0.01

model_order = [s for s in ["gru", "gru_transformer", "transformer", "mamba"] if s in set(models["model_type"])]
model_order.extend(sorted(s for s in models["model_type"].unique() if s not in model_order))
plot_data = [models.loc[models["model_type"].eq(m), "improvement_bps"].dropna().to_numpy() for m in model_order]

fig, ax = plt.subplots(figsize=(8.2, 5.4))
box = ax.boxplot(
    plot_data,
    positions=np.arange(len(model_order)),
    widths=0.48,
    patch_artist=True,
    showfliers=False,
    medianprops={"color": "black", "linewidth": 1.2},
    whiskerprops={"color": "0.35", "linewidth": 1.0},
    capprops={"color": "0.35", "linewidth": 1.0},
)
palette = plt.get_cmap("tab10")
for i, patch in enumerate(box["boxes"]):
    patch.set_facecolor(palette(i % 10))
    patch.set_alpha(0.22)
    patch.set_edgecolor(palette(i % 10))
    patch.set_linewidth(1.2)

rng = np.random.default_rng(7)
for i, model_type in enumerate(model_order):
    group = models[models["model_type"].eq(model_type)].sort_values("ticker")
    jitter = rng.normal(0.0, 0.045, size=len(group))
    ax.scatter(
        np.full(len(group), i) + jitter,
        group["improvement_bps"],
        s=38,
        color=palette(i % 10),
        edgecolor="white",
        linewidth=0.6,
        alpha=0.86,
        zorder=3,
    )
    mean_value = group["improvement_bps"].mean()
    # ax.scatter(i, mean_value, marker="D", s=58, color="black", zorder=4, label="Mean" if i == 0 else None)

ax.axhline(0.0, color="0.25", linewidth=1.0, linestyle="--", alpha=0.8)
ax.set_yscale("symlog", linthresh=linthresh, linscale=0.75)
ax.set_xticks(np.arange(len(model_order)))
ax.set_xticklabels([STRATEGY_LABELS.get(m, m) for m in model_order], ha="center")
ax.set_ylabel(f"Improvement vs baseline ({metric_label(SELECTED_METRIC)})")
ax.set_title(f"{PRIMARY_RUN_MODE}: cross-ticker improvement over baseline")
ax.text(
    0.99,
    0.02,
    f"symlog y-axis, linear within +/-{linthresh:.3g} bps",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=8,
    color="0.45",
)
ax.grid(axis="y", alpha=0.25)
if ax.get_legend_handles_labels()[0]:
    ax.legend(frameon=False, loc="best")
fig.tight_layout()
plt.show()

stability_summary = (
    models.groupby("model_type")["improvement_bps"]
    .agg(mean="mean", median="median", std="std", min="min", max="max", count="count")
    .reindex(model_order)
    .reset_index()
)
display(stability_summary)


In [ ]:
# Paired ticker-level p-values for improvement over baseline.
# Positive paired difference means the model has lower selected metric than baseline.

def sign_flip_pvalue(diff, *, alternative="greater"):
    diff = np.asarray(diff, dtype=float)
    diff = diff[np.isfinite(diff)]
    n = len(diff)
    if n == 0:
        return np.nan
    observed = diff.mean()
    signs = np.array(np.meshgrid(*[[-1, 1]] * n)).T.reshape(-1, n)
    null = (signs * diff).mean(axis=1)
    if alternative == "greater":
        return float((np.sum(null >= observed) + 1) / (len(null) + 1))
    if alternative == "less":
        return float((np.sum(null <= observed) + 1) / (len(null) + 1))
    extreme = np.abs(null) >= abs(observed)
    return float((np.sum(extreme) + 1) / (len(null) + 1))


def holm_adjust(p_values):
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)
    valid = np.where(np.isfinite(p_values))[0]
    if len(valid) == 0:
        return adjusted
    order = valid[np.argsort(p_values[valid])]
    running = 0.0
    m = len(order)
    for rank, idx in enumerate(order):
        value = min(1.0, (m - rank) * p_values[idx])
        running = max(running, value)
        adjusted[idx] = running
    return adjusted


test_df = analysis_df.copy()
if SELECTED_METRIC not in test_df.columns:
    raise ValueError(f"Metric {SELECTED_METRIC!r} is not available.")
test_df[SELECTED_METRIC] = pd.to_numeric(test_df[SELECTED_METRIC], errors="coerce")
test_df["model_type"] = test_df["model_type"].astype(str)
wide = test_df.pivot_table(index="ticker", columns="model_type", values=SELECTED_METRIC, aggfunc="first")
if "baseline" not in wide.columns:
    raise ValueError("No baseline rows found; cannot compute paired improvement p-values.")

rows = []
model_order_for_tests = [m for m in ["gru", "gru_transformer", "transformer", "mamba"] if m in wide.columns]
model_order_for_tests.extend(sorted(m for m in wide.columns if m not in ["baseline", *model_order_for_tests]))
for model_type in model_order_for_tests:
    paired = wide[["baseline", model_type]].dropna()
    diff = paired["baseline"] - paired[model_type]
    rows.append({
        "model_type": model_type,
        "metric": SELECTED_METRIC,
        "n_tickers": int(len(diff)),
        "mean_improvement_bps": float(diff.mean()) if len(diff) else np.nan,
        "median_improvement_bps": float(diff.median()) if len(diff) else np.nan,
        "positive_tickers": int((diff > 0).sum()),
        "negative_tickers": int((diff < 0).sum()),
        "zero_tickers": int((diff == 0).sum()),
        "sign_flip_p_one_sided": sign_flip_pvalue(diff, alternative="greater"),
    })

pvalue_table = pd.DataFrame(rows)
pvalue_table["holm_p_one_sided"] = holm_adjust(pvalue_table["sign_flip_p_one_sided"].to_numpy())
display(pvalue_table)


In [ ]:
# Pairwise model-vs-model p-value matrix across tickers.
# Entry (row, column) tests whether the row model has lower selected metric
# than the column model. For toxic/IS metrics, lower is better.

# Reuse sign_flip_pvalue and holm_adjust from the previous p-value cell if available.
if "sign_flip_pvalue" not in globals():
    def sign_flip_pvalue(diff, *, alternative="greater"):
        diff = np.asarray(diff, dtype=float)
        diff = diff[np.isfinite(diff)]
        n = len(diff)
        if n == 0:
            return np.nan
        observed = diff.mean()
        signs = np.array(np.meshgrid(*[[-1, 1]] * n)).T.reshape(-1, n)
        null = (signs * diff).mean(axis=1)
        if alternative == "greater":
            return float((np.sum(null >= observed) + 1) / (len(null) + 1))
        if alternative == "less":
            return float((np.sum(null <= observed) + 1) / (len(null) + 1))
        extreme = np.abs(null) >= abs(observed)
        return float((np.sum(extreme) + 1) / (len(null) + 1))

if "holm_adjust" not in globals():
    def holm_adjust(p_values):
        p_values = np.asarray(p_values, dtype=float)
        adjusted = np.full(len(p_values), np.nan)
        valid = np.where(np.isfinite(p_values))[0]
        if len(valid) == 0:
            return adjusted
        order = valid[np.argsort(p_values[valid])]
        running = 0.0
        m = len(order)
        for rank, idx in enumerate(order):
            value = min(1.0, (m - rank) * p_values[idx])
            running = max(running, value)
            adjusted[idx] = running
        return adjusted

pairwise_df = analysis_df.copy()
if SELECTED_METRIC not in pairwise_df.columns:
    raise ValueError(f"Metric {SELECTED_METRIC!r} is not available.")
pairwise_df[SELECTED_METRIC] = pd.to_numeric(pairwise_df[SELECTED_METRIC], errors="coerce")
pairwise_df["model_type"] = pairwise_df["model_type"].astype(str)
wide_models = pairwise_df.pivot_table(
    index="ticker",
    columns="model_type",
    values=SELECTED_METRIC,
    aggfunc="first",
)

PAIRWISE_MODEL_ORDER = ["gru", "gru_transformer", "transformer", "mamba"]
pairwise_models = [m for m in PAIRWISE_MODEL_ORDER if m in wide_models.columns]
pairwise_models.extend(sorted(m for m in wide_models.columns if m not in ["baseline", *pairwise_models]))

rows = []
raw_matrix = pd.DataFrame(np.nan, index=pairwise_models, columns=pairwise_models)
mean_diff_matrix = pd.DataFrame(np.nan, index=pairwise_models, columns=pairwise_models)
n_matrix = pd.DataFrame(np.nan, index=pairwise_models, columns=pairwise_models)
for row_model in pairwise_models:
    for col_model in pairwise_models:
        if row_model == col_model:
            raw_matrix.loc[row_model, col_model] = np.nan
            mean_diff_matrix.loc[row_model, col_model] = 0.0
            continue
        paired = wide_models[[row_model, col_model]].dropna()
        # Positive diff means row_model is better because lower metric is better.
        diff = paired[col_model] - paired[row_model]
        p_value = sign_flip_pvalue(diff, alternative="greater")
        raw_matrix.loc[row_model, col_model] = p_value
        mean_diff_matrix.loc[row_model, col_model] = float(diff.mean()) if len(diff) else np.nan
        n_matrix.loc[row_model, col_model] = int(len(diff))
        rows.append({
            "row_model": row_model,
            "column_model": col_model,
            "metric": SELECTED_METRIC,
            "n_tickers": int(len(diff)),
            "mean_row_improvement_vs_column_bps": float(diff.mean()) if len(diff) else np.nan,
            "median_row_improvement_vs_column_bps": float(diff.median()) if len(diff) else np.nan,
            "positive_tickers": int((diff > 0).sum()),
            "negative_tickers": int((diff < 0).sum()),
            "zero_tickers": int((diff == 0).sum()),
            "p_one_sided": p_value,
        })

pairwise_pvalues = pd.DataFrame(rows)
pairwise_pvalues["holm_p_one_sided_all_ordered_pairs"] = holm_adjust(
    pairwise_pvalues["p_one_sided"].to_numpy()
)

# Often the claim of interest is specifically Mamba vs the other encoders.
mamba_mask = pairwise_pvalues["row_model"].eq("mamba")
pairwise_pvalues["holm_p_one_sided_mamba_vs_others"] = np.nan
pairwise_pvalues.loc[mamba_mask, "holm_p_one_sided_mamba_vs_others"] = holm_adjust(
    pairwise_pvalues.loc[mamba_mask, "p_one_sided"].to_numpy()
)

display(
    raw_matrix.style
    .format("{:.4f}", na_rep="--")
    .set_caption(
        f"Raw one-sided pairwise p-values for {SELECTED_METRIC}; "
        "entry (row, column) tests row model < column model"
    )
)
display(
    mean_diff_matrix.style
    .format("{:.4f}", na_rep="--")
    .set_caption(
        f"Mean paired improvement in bps for {SELECTED_METRIC}; "
        "positive means row model is better than column model"
    )
)
display(pairwise_pvalues)
